# Notebook 05: Training — NASH Condition

**Agency Calculus Empirical Validation — Paper C**

Social planner objective: **R = Σ log(u_i + ε),  ε = 1e-4**

Nash social welfare. Permits bounded compensation: the log curvature
means the exchange rate between agents is bounded, so floor compression
is costly but not infinitely so.

**Epsilon is fine for NASH.** Nash honestly permits compensation — epsilon
only provides numerical stability, not structural modification.

**Predicted outcomes:**
- Floor utility: stable or slightly improving (log curvature discourages compression)
- Gini coefficient: moderate (lower than SUM, higher than JAM)
- Tax on floor agent: moderate
- Total utility: high but below SUM

5 seeds × 10M steps each.

In [ ]:
import sys, os
import numpy as np

sys.path.insert(0, os.path.join(os.getcwd(), '..', 'src'))

from training import run_training, TOTAL_TIMESTEPS, N_SEEDS
from metrics import MetricsLogger
import matplotlib.pyplot as plt

CONDITION = 'nash'
RESULTS_DIR = '../results'
os.makedirs(RESULTS_DIR, exist_ok=True)
print(f'Condition: {CONDITION}  (R = Σ log(u_i + 1e-4))')

In [ ]:
# Run a single seed (set SEED = 0, 1, 2, 3, 4 across separate sessions)
SEED = 0  # Change this for each session

save_path = f'{RESULTS_DIR}/{CONDITION}_seed{SEED}_metrics.npz'
if os.path.exists(save_path):
    print(f'Seed {SEED} already complete: {save_path}')
else:
    print(f'Starting training: condition={CONDITION}, seed={SEED}')
    logger = run_training(
        condition=CONDITION,
        seed=SEED,
        total_timesteps=TOTAL_TIMESTEPS,
        results_dir=RESULTS_DIR,
    )
    print(f'Training complete. Saved to {save_path}')

In [ ]:
# DEBUG: short run
# logger_debug = run_training(condition=CONDITION, seed=99,
#     total_timesteps=1_000_000, results_dir=RESULTS_DIR)
print('Debug run commented out. Uncomment to verify.')

In [ ]:
def plot_seed_progress(condition, seed, results_dir=RESULTS_DIR):
    path = f'{results_dir}/{condition}_seed{seed}_metrics.npz'
    if not os.path.exists(path):
        return
    logger = MetricsLogger.load(path)
    arrays = logger.to_arrays()
    steps = arrays.get('step', np.array([]))
    metrics_to_plot = ['floor_utility_mean', 'total_utility_mean', 'gini_wealth_mean']
    fig, axes = plt.subplots(1, 3, figsize=(14, 4))
    for ax, metric in zip(axes, metrics_to_plot):
        if metric in arrays:
            ax.plot(steps, arrays[metric], color='#f39c12', linewidth=2)
            ax.set_title(metric)
            ax.set_xlabel('Training Steps')
            ax.spines['top'].set_visible(False)
            ax.spines['right'].set_visible(False)
    plt.suptitle(f'NASH Condition — Seed {seed}', y=1.02)
    plt.tight_layout()
    plt.show()

for s in range(N_SEEDS):
    plot_seed_progress(CONDITION, s)

## Notes on Expected Results

NASH should produce intermediate outcomes between SUM and JAM:
- The log curvature makes large utility disparities costly
- But bounded compensation means moderate floor compression still happens
- The planner will protect the floor agent somewhat but not absolutely

The key comparison with JAM: NASH floor utility should be lower and NASH Gini
should be higher. If NASH and JAM produce identical outcomes, something is
wrong with the JAM singularity preservation.

**Next:** Notebook 06 — JAM condition (the main theoretical prediction).